In [1]:
# Load vehicles_clean.parquet (not the Excel — build on the cleaned version), 
# import re for regex.

import pandas as pd
import numpy as np
import re

pd.set_option('display.max_columns', 100)

df = pd.read_parquet('../data/processed/vehicles_clean.parquet')
print(df.shape)

(50242, 35)


In [2]:
# Inspect exact category spellings in fuelType1, fuelType2, atvType 
# before writing classification logic against them.

print(df['fuelType1'].value_counts(dropna=False))
print()
print(df['fuelType2'].value_counts(dropna=False))
print()
print(df['atvType'].value_counts(dropna=False))

fuelType1
Regular Gasoline     31324
Premium Gasoline     15759
Electricity           1572
Diesel                1310
Midgrade Gasoline      173
Natural Gas             60
Hydrogen                42
NaN                      2
Name: count, dtype: int64

fuelType2
NaN            48229
E85             1538
Electricity      447
Natural Gas       20
Propane            8
Name: count, dtype: int64

atvType
NaN               43452
Hybrid             1873
EV                 1572
FFV                1538
Diesel             1238
Plug-in Hybrid      447
CNG                  50
FCV                  42
Bifuel (CNG)         20
Bifuel (LPG)          8
eFCV                  2
Name: count, dtype: int64


In [3]:
# Build powertrain (Gasoline/Diesel/HEV/PHEV/BEV/CNG/Flex-Fuel) via a row-wise
# function that trusts atvType first, falls back to fuelType1/fuelType2.

def classify_powertrain(row):
    atv = row['atvType']
    ft1 = row['fuelType1']
    ft2 = row['fuelType2']

    if pd.notna(atv):
        atv_lower = str(atv).lower()
        if 'plug-in' in atv_lower:
            return 'PHEV'
        if atv_lower == 'ev':
            return 'BEV'
        if 'hybrid' in atv_lower:
            return 'HEV'
        if 'cng' in atv_lower:
            return 'CNG'
        if 'ffv' in atv_lower:
            return 'Flex-Fuel'
        if 'diesel' in atv_lower:
            return 'Diesel'

    if pd.notna(ft1) and 'electricity' in str(ft1).lower() and pd.isna(ft2):
        return 'BEV'

    if pd.notna(ft1) and 'diesel' in str(ft1).lower():
        return 'Diesel'

    fuel1 = str(ft1).lower() if pd.notna(ft1) else ''
    fuel2 = str(ft2).lower() if pd.notna(ft2) else ''
    if 'electricity' in fuel2:
        return 'PHEV'
    if 'hydrogen' in fuel1:
        return 'Hydrogen'
    if 'natural gas' in fuel1 or 'cng' in fuel1 or 'cng' in fuel2:
        return 'CNG'
    if 'lpg' in fuel1 or 'lpg' in fuel2 or 'propane' in fuel1:
        return 'LPG'
    if 'e85' in fuel1 or 'e85' in fuel2:
        return 'Flex-Fuel'
    if 'gasoline' in fuel1:
        return 'Gasoline'
    return 'Other'

df['powertrain'] = df.apply(classify_powertrain, axis=1)
print(df['powertrain'].value_counts())

powertrain
Gasoline     43378
HEV           1873
BEV           1572
Flex-Fuel     1538
Diesel        1310
PHEV           447
CNG             80
Hydrogen        42
Other            2
Name: count, dtype: int64


In [4]:
# Validate powertrain with cross-tabs against the source columns and
# a year-by-year breakdown (electrified types should be ~absent pre-2000s).

pd.crosstab(df['powertrain'], df['atvType'], dropna=False)
pd.crosstab(df['powertrain'], df['fuelType1'], dropna=False)

fuelType1,Diesel,Electricity,Hydrogen,Midgrade Gasoline,Natural Gas,Premium Gasoline,Regular Gasoline,NaN
powertrain,,,,,,,,
BEV,0,1572,0,0,0,0,0,0
CNG,0,0,0,0,60,0,20,0
Diesel,1310,0,0,0,0,0,0,0
Flex-Fuel,0,0,0,0,0,128,1410,0
Gasoline,0,0,0,154,0,14364,28860,0
HEV,0,0,0,19,0,956,898,0
Hydrogen,0,0,42,0,0,0,0,0
Other,0,0,0,0,0,0,0,2
PHEV,0,0,0,0,0,311,136,0


In [5]:
df.groupby('year')['powertrain'].value_counts().unstack(fill_value=0).tail(15)

powertrain,BEV,CNG,Diesel,Flex-Fuel,Gasoline,HEV,Hydrogen,Other,PHEV
year,,,,,,,,,
2013,14,1,17,158,935,51,0,0,4
2014,15,2,30,140,972,52,1,0,10
2015,18,2,36,98,1068,46,1,0,12
2016,31,1,26,69,1071,46,2,0,18
2017,30,0,21,64,1116,43,3,0,19
2018,24,0,38,54,1153,45,2,0,34
2019,35,0,30,43,1136,63,4,0,36
2020,38,0,20,24,1044,83,4,0,45
2021,51,0,30,15,1009,126,5,0,50


In [6]:
# Inspect BEV engine fields: most are missing in this snapshot.
# Preserve recorded zeros; do not impute missing values as zero.

df[df['powertrain'] == 'BEV'][['cylinders', 'displ']].describe()

,cylinders,displ
count,0.0,1.0
mean,NaN,0.0
std,NaN,NaN
min,NaN,0.0
25%,NaN,0.0
50%,NaN,0.0
75%,NaN,0.0
max,NaN,0.0


In [7]:
df[df['powertrain'] != 'BEV'][['cylinders', 'displ']].eq(0).sum()

cylinders    0
displ        0
dtype: int64

In [8]:
# Parse trany into transmission_type (Manual/Automatic/CVT) and num_gears via regex.

def parse_transmission_type(trany):
    if pd.isna(trany):
        return np.nan

    t = trany.lower()

    if 'variable gear ratios' in t:
        return 'CVT'
    if t.startswith('manual'):
        return 'Manual'
    if t.startswith('automatic'):
        return 'Automatic'

    return np.nan


def parse_num_gears(trany):
    if pd.isna(trany):
        return np.nan

    # Examples: Automatic (AM5), Automatic (AM6), Automatic (L4), Automatic (A2)
    match = re.search(r'\(([A-Z]+)(\d+)\)', trany)
    if match:
        return int(match.group(2))

    # Examples: 5-spd, S6
    match = re.search(r'(\d+)-spd', trany)
    if match:
        return int(match.group(1))

    match = re.search(r'S(\d+)', trany)
    if match:
        return int(match.group(1))

    # CVT has variable gear ratios, so no fixed gear count
    return np.nan


df['transmission_type'] = df['trany'].apply(parse_transmission_type)
df['num_gears'] = df['trany'].apply(parse_num_gears)

In [9]:
# Validate the parsing — check value counts and find any trany strings 
# that didn't match either regex pattern.

df['transmission_type'].value_counts(dropna=False)

transmission_type
Automatic    35727
Manual       13297
CVT           1207
NaN             11
Name: count, dtype: int64

In [10]:
df['num_gears'].value_counts(dropna=False).sort_index()

num_gears
1.0      1556
2.0       103
3.0      3230
4.0     12783
5.0     11457
6.0      9072
7.0      2689
8.0      5584
9.0      1225
10.0     1325
NaN      1218
Name: count, dtype: int64

In [11]:
missed = df[df['transmission_type'].notna() & df['num_gears'].isna()]
print(missed['trany'].unique())

<ArrowStringArray>
['Automatic (variable gear ratios)']
Length: 1, dtype: str


In [12]:
# Collapse the 30+ VClass categories into 7 simplified segment buckets, 
# validated with a cross-tab.

def simplify_segment(vclass):
    v = str(vclass).lower()
    if 'pickup' in v:
        return 'Truck/Pickup'
    if 'sport utility' in v or 'suv' in v:
        return 'SUV'
    if 'van' in v:
        return 'Van/Minivan'
    if 'two seater' in v:
        return 'Sports/Two-Seater'
    if 'station wagon' in v:
        return 'Wagon'
    if 'special purpose' in v:
        return 'Special Purpose'
    return 'Car'

df['segment'] = df['VClass'].apply(simplify_segment)
pd.crosstab(df['segment'], df['VClass'])

VClass,Compact Cars,Large Cars,Midsize Cars,Midsize Station Wagons,Midsize-Large Station Wagons,Minicompact Cars,Minivan - 2WD,Minivan - 4WD,Small Pickup Trucks,Small Pickup Trucks 2WD,Small Pickup Trucks 4WD,Small Sport Utility Vehicle 2WD,Small Sport Utility Vehicle 4WD,Small Station Wagons,Special Purpose Vehicle,Special Purpose Vehicle 2WD,Special Purpose Vehicle 4WD,Special Purpose Vehicles,Special Purpose Vehicles/2wd,Special Purpose Vehicles/4wd,Sport Utility Vehicle - 2WD,Sport Utility Vehicle - 4WD,Standard Pickup Trucks,Standard Pickup Trucks 2WD,Standard Pickup Trucks 4WD,Standard Pickup Trucks/2wd,Standard Sport Utility Vehicle 2WD,Standard Sport Utility Vehicle 4WD,Subcompact Cars,Two Seaters,Vans,Vans Passenger,"Vans, Cargo Type","Vans, Passenger Type"
segment,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Car,6625,2819,5886,0,0,1671,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,5822,0,0,0,0,0
SUV,0,0,0,0,0,0,0,0,0,0,0,1221,2292,0,0,0,0,0,0,0,1627,2078,0,0,0,0,621,2066,0,0,0,0,0,0
Special Purpose,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,748,356,1455,2,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Sports/Two-Seater,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2531,0,0,0,0
Truck/Pickup,0,0,0,0,0,0,0,0,538,519,337,0,0,0,0,0,0,0,0,0,0,0,2354,1497,1724,4,0,0,0,0,0,0,0,0
Van/Minivan,0,0,0,0,0,0,411,64,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1141,2,438,323
Wagon,0,0,0,618,656,0,0,0,0,0,0,0,0,1793,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [13]:
# Add decade and a post_2012 boolean flag for the CAFE-era comparison used in Step 5.

df['decade'] = (df['year'] // 10) * 10
df['post_2012'] = df['year'] >= 2012

In [14]:
# Fix UCity/UHighway 0-as-missing, then compute epa_gap_city/epa_gap_hwy:
# These are unadjusted laboratory ratings minus adjusted published ratings.
# They are not owner-reported or observed real-world fuel economy.

df['UCity'] = df['UCity'].replace(0, np.nan)
df['UHighway'] = df['UHighway'].replace(0, np.nan)

df['epa_gap_city'] = df['UCity'] - df['city08']
df['epa_gap_hwy'] = df['UHighway'] - df['highway08']

df[['epa_gap_city', 'epa_gap_hwy']].describe().round(2)

,epa_gap_city,epa_gap_hwy
count,50217.00,50217.00
mean,6.05,10.90
std,6.97,5.78
min,-21.27,-63.45
25%,3.66,8.00
50%,4.40,9.85
75%,5.83,12.00
max,74.80,64.00


In [15]:
# Save to data/processed/vehicles_features.parquet.

df.to_parquet('../data/processed/vehicles_features.parquet', index=False)
print("Saved:", df.shape)
print(df.columns.tolist())

Saved: (50242, 43)
['id', 'year', 'make', 'model', 'VClass', 'drive', 'trany', 'cylinders', 'displ', 'fuelType', 'fuelType1', 'fuelType2', 'atvType', 'city08', 'highway08', 'comb08', 'UCity', 'UHighway', 'cityA08', 'highwayA08', 'combA08', 'fuelCost08', 'co2TailpipeGpm', 'mpgData', 'startStop', 'phevCity', 'phevHwy', 'phevComb', 'range', 'rangeCity', 'rangeHwy', 'rangeA', 'charge120', 'charge240', 'evMotor', 'powertrain', 'transmission_type', 'num_gears', 'segment', 'decade', 'post_2012', 'epa_gap_city', 'epa_gap_hwy']


In [16]:
# Build a human-readable column mapping for the Excel export (raw name -> friendly name)
friendly_cols = {
    'year': 'Model Year', 'make': 'Make', 'model': 'Model',
    'segment': 'Vehicle Segment', 'VClass': 'EPA Vehicle Class',
    'drive': 'Drivetrain', 'transmission_type': 'Transmission Type',
    'num_gears': 'Number of Gears', 'cylinders': 'Cylinders',
    'displ': 'Engine Size (L)', 'powertrain': 'Powertrain Type',
    'city08': 'City MPG / MPGe', 'highway08': 'Highway MPG / MPGe', 'comb08': 'Combined MPG / MPGe',
    'fuelCost08': 'Estimated Annual Fuel Cost ($)', 'co2TailpipeGpm': 'CO2 Emissions (g/mile)'
}

# Select and rename only the columns worth showing to a non-technical reader
friendly = df[list(friendly_cols.keys())].rename(columns=friendly_cols)
print(friendly.shape)
friendly.head()

(50242, 16)


,Model Year,Make,Model,Vehicle Segment,EPA Vehicle Class,Drivetrain,Transmission Type,Number of Gears,Cylinders,Engine Size (L),Powertrain Type,City MPG / MPGe,Highway MPG / MPGe,Combined MPG / MPGe,Estimated Annual Fuel Cost ($),CO2 Emissions (g/mile)
0,1985,Alfa Romeo,Spider Veloce 2000,Sports/Two-Seater,Two Seaters,Rear-Wheel Drive,Manual,5.0,4.0,2.0,Gasoline,19,25,21,2950,423.190476
1,1985,Ferrari,Testarossa,Sports/Two-Seater,Two Seaters,Rear-Wheel Drive,Manual,5.0,12.0,4.9,Gasoline,9,14,11,5650,807.909091
2,1985,Dodge,Charger,Car,Subcompact Cars,Front-Wheel Drive,Manual,5.0,4.0,2.2,Gasoline,23,33,27,2300,329.148148
3,1985,Dodge,B150/B250 Wagon 2WD,Van/Minivan,Vans,Rear-Wheel Drive,Automatic,3.0,8.0,5.2,Gasoline,10,12,11,5650,807.909091
4,1993,Subaru,Legacy AWD Turbo,Car,Compact Cars,4-Wheel or All-Wheel Drive,Manual,5.0,4.0,2.2,Gasoline,17,23,19,4100,467.736842


In [17]:
# Build a small data dictionary table describing each friendly column, to include as a second sheet
data_dictionary = pd.DataFrame({
    'Column': list(friendly_cols.values()),
    'Description': [
        'The vehicle model year',
        'Vehicle manufacturer',
        'Specific model name',
        'Simplified vehicle category (Car, SUV, Truck/Pickup, etc.)',
        'Original EPA vehicle class (more granular)',
        'Front-wheel, rear-wheel, all-wheel, or 4-wheel drive',
        'Manual, Automatic, or CVT',
        'Number of gears (blank for CVT, which has none)',
        'Engine cylinder count; missing values retained, including most BEVs',
        'Engine displacement in liters; missing values retained, including most BEVs',
        'Gasoline, Diesel, Hybrid (HEV), Plug-in Hybrid (PHEV), Battery Electric (BEV), etc.',
        'EPA city fuel economy (MPG, or MPGe for electric vehicles)',
        'EPA highway fuel economy (MPG, or MPGe for electric vehicles)',
        'EPA combined city/highway fuel economy (MPG, or MPGe for electric vehicles)',
        'EPA-estimated annual fuel cost in dollars, based on assumed national average fuel prices',
        'Tailpipe CO2 emissions in grams per mile'
    ]
})
data_dictionary

,Column,Description
0,Model Year,The vehicle model year
1,Make,Vehicle manufacturer
2,Model,Specific model name
3,Vehicle Segment,"Simplified vehicle category (Car, SUV, Truck/P..."
4,EPA Vehicle Class,Original EPA vehicle class (more granular)
5,Drivetrain,"Front-wheel, rear-wheel, all-wheel, or 4-wheel..."
6,Transmission Type,"Manual, Automatic, or CVT"
7,Number of Gears,"Number of gears (blank for CVT, which has none)"
8,Cylinders,Engine cylinder count; missing values retained...
9,Engine Size (L),Engine displacement in liters; missing values ...


In [18]:
# Write both sheets into one formatted Excel workbook
with pd.ExcelWriter('../data/processed/vehicles_clean.xlsx', engine='openpyxl') as writer:
    friendly.to_excel(writer, sheet_name='Vehicle Data', index=False)
    data_dictionary.to_excel(writer, sheet_name='Data Dictionary', index=False)

In [19]:
# Post-process formatting: bold header row, frozen header, auto-sized columns
from openpyxl import load_workbook
from openpyxl.styles import Font
from openpyxl.utils import get_column_letter

file_path = '../data/processed/vehicles_clean.xlsx'

wb = load_workbook(file_path)
ws = wb['Vehicle Data']

# Freeze header row
ws.freeze_panes = 'A2'

# Bold header row
for cell in ws[1]:
    cell.font = Font(bold=True)

# Auto-size columns
for i, col in enumerate(friendly.columns, 1):
    max_len = max(
        max(len(str(value)) for value in friendly[col]),
        len(str(col))
    ) + 2

    ws.column_dimensions[get_column_letter(i)].width = min(max_len, 30)

# Save formatted workbook
wb.save(file_path)

print("Saved clean Excel workbook.")

Saved clean Excel workbook.
